# ForestWatch Papua — Optimalisasi Dataset (Subsampling Kelas Dominan)

Notebook ini **tidak memodifikasi file patch asli**. Tujuannya:
1. Scan distribusi piksel per kelas dari seluruh patch training
2. Subsample patch yang didominasi **Perairan** dan **Hutan** agar dataset lebih seimbang
3. Tampilkan EDA distribusi **sebelum** dan **sesudah** normalisasi (teks, tanpa grafik)
4. Simpan daftar patch terpilih ke cache untuk dipakai di notebook training

**Target pengurangan:**
| Kelas | Sebelum | Target |
|---|---|---|
| Perairan (0) | ~6.86 B px | 800 juta px |
| Hutan (1) | ~3.71 B px | 1 miliar px |
| Kelas lain | keep all | keep all |

## Bagian 0 — Setup environment (Colab / Lab / Jetson)

In [1]:
# === Bagian 0 — Setup ===
ENV = "colab"   # "colab" | "lab" | "jetson"
from pathlib import Path

DRIVE_ROOT = None
DATA_ROOT  = None

if ENV == "colab":
    import subprocess, sys, importlib
    subprocess.run(
        "cd /content && (git -C fw_repo pull -q || git clone --depth 1 "
        "https://github.com/Ridho-Dwi-Syahputra/forestwatch-model.git fw_repo)",
        shell=True, check=False,
    )
    subprocess.run("pip install -q -e /content/fw_repo[gee,gis,ml]", shell=True, check=False)
    if "/content/fw_repo/src" not in sys.path:
        sys.path.insert(0, "/content/fw_repo/src")
    for _m in [m for m in list(sys.modules) if m == "forestwatch" or m.startswith("forestwatch.")]:
        del sys.modules[_m]
    importlib.invalidate_caches()
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_ROOT = Path("/content/drive/MyDrive/Satria Data 3.0")

elif ENV == "lab":
    DRIVE_ROOT = Path("G:/My Drive/Satria Data 3.0")  # sesuaikan path Drive Desktop

elif ENV == "jetson":
    DATA_ROOT = Path("/mnt/forestwatch_dataset")
    assert DATA_ROOT.exists(), f"{DATA_ROOT} tidak ada — mount SMB share dari PC dulu."

print(f"ENV={ENV} | DRIVE_ROOT={DRIVE_ROOT} | DATA_ROOT={DATA_ROOT}")

Mounted at /content/drive
ENV=colab | DRIVE_ROOT=/content/drive/MyDrive/Satria Data 3.0 | DATA_ROOT=None


In [2]:
# === Path sumber patch + load daftar file ===
import json, random
import numpy as np
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm
from forestwatch.constants import N_CLASSES, CLASS_NAMES
from forestwatch.data import list_patches, split_files

if ENV == "jetson":
    BAHAN_DIR  = DATA_ROOT
    CACHE_DIR  = DATA_ROOT / "optimize_cache"
    train_files = list_patches(BAHAN_DIR / "train")
    val_files   = list_patches(BAHAN_DIR / "val")
    test_files  = list_patches(BAHAN_DIR / "test")
else:
    PATCH_DIR         = DRIVE_ROOT / "ForestWatch_Patches"
    PATCHES_TRANSFER  = DRIVE_ROOT / "ForestWatch_Patches_Transfer"
    AUGMENTED_PATCHES = DRIVE_ROOT / "Augmented_Patches"
    BAHAN_DIR         = DRIVE_ROOT / "Bahan_Training_Model"
    CACHE_DIR         = BAHAN_DIR  / "optimize_cache"

    papua_files    = list_patches(PATCH_DIR)
    transfer_files = list_patches(PATCHES_TRANSFER)
    aug_files      = list_patches(AUGMENTED_PATCHES)
    train_p, val_files, test_files = split_files(papua_files, train_ratio=0.8, val_ratio=0.1, seed=42)
    train_files = list(train_p) + list(transfer_files) + list(aug_files)

CACHE_DIR.mkdir(parents=True, exist_ok=True)
print(f"train={len(train_files):,}  val={len(val_files):,}  test={len(test_files):,}")
print(f"CLASS_NAMES: {CLASS_NAMES}")

train=173,078  val=19,345  test=19,346
CLASS_NAMES: ('Perairan', 'Hutan', 'Lahan Terbuka', 'Sawit', 'Pertanian Lain', 'Tambang', 'Permukiman')


In [3]:
# === Scan distribusi piksel per patch (cached — skip jika sudah ada) ===
SCAN_CACHE = CACHE_DIR / "patch_class_counts.json"

def _count_patch(path):
    data = np.load(path)
    lab  = data["lab"].flatten().astype(int)
    return np.bincount(lab, minlength=N_CLASSES).tolist()

def scan_patches(files, cache_path, max_workers=64):
    cache_p = Path(cache_path)
    dist    = {}
    if cache_p.exists():
        print(f"[cache] memuat {cache_p.name} ...")
        dist = json.loads(cache_p.read_text(encoding="utf-8"))
        missing = [f for f in files if str(f) not in dist]
        if not missing:
            print(f"  semua {len(files):,} patch sudah di cache.")
            return dist
        print(f"  {len(missing):,} patch belum ada di cache, scan sekarang ...")
        files_to_scan = missing
    else:
        files_to_scan = files

    with ThreadPoolExecutor(max_workers=max_workers) as exe:
        futures = {exe.submit(_count_patch, f): str(f) for f in files_to_scan}
        for fut in tqdm(as_completed(futures), total=len(futures), desc="Scanning patches", unit="patch"):
            dist[futures[fut]] = fut.result()

    cache_p.write_text(json.dumps(dist), encoding="utf-8")
    print(f"[cache] disimpan ke {cache_p}")
    return dist

patch_dist = scan_patches(train_files, SCAN_CACHE, max_workers=64)
print(f"\nTotal patch terscan: {len(patch_dist):,}")

Scanning patches:   0%|          | 0/173078 [00:00<?, ?patch/s]

[cache] disimpan ke /content/drive/MyDrive/Satria Data 3.0/Bahan_Training_Model/optimize_cache/patch_class_counts.json

Total patch terscan: 173,078


In [4]:
# === EDA SEBELUM normalisasi ===

def print_dist_table(files, patch_dist, label):
    total_px  = [0] * N_CLASSES
    dom_count = [0] * N_CLASSES
    for f in files:
        counts = patch_dist.get(str(f), [0] * N_CLASSES)
        for c in range(N_CLASSES):
            total_px[c] += counts[c]
        dom_count[int(np.argmax(counts))] += 1
    grand = sum(total_px)

    print(f"\n{'='*74}")
    print(f"  {label}")
    print(f"  Total patches : {len(files):,}")
    print(f"  Total piksel  : {grand:,}")
    print(f"{'='*74}")
    print(f"  {'Kelas':<18} {'Piksel':>18} {'%':>7}   {'Patch (dominan)':>15}")
    print(f"  {'-'*62}")
    for c in range(N_CLASSES):
        pct = 100 * total_px[c] / grand if grand else 0
        print(f"  {CLASS_NAMES[c]:<18} {total_px[c]:>18,} {pct:>6.1f}%   {dom_count[c]:>12,}")
    print(f"  {'-'*62}")
    print(f"  {'TOTAL':<18} {grand:>18,}  100.0%   {len(files):>12,}")
    print(f"{'='*74}")

print_dist_table(train_files, patch_dist, "SEBELUM normalisasi — Train FINAL (papua + transfer + aug)")


  SEBELUM normalisasi — Train FINAL (papua + transfer + aug)
  Total patches : 173,078
  Total piksel  : 11,342,839,808
  Kelas                          Piksel       %   Patch (dominan)
  --------------------------------------------------------------
  Perairan                6,857,785,545   60.5%        103,680
  Hutan                   3,714,769,070   32.7%         60,047
  Lahan Terbuka             138,579,237    1.2%            774
  Sawit                     126,185,654    1.1%          1,604
  Pertanian Lain            297,497,904    2.6%          3,675
  Tambang                    82,216,580    0.7%          1,369
  Permukiman                125,805,818    1.1%          1,929
  --------------------------------------------------------------
  TOTAL                  11,342,839,808  100.0%        173,078


In [5]:
# === Subsampling: kurangi patch dominan Perairan & Hutan ke target piksel ===

PIXEL_TARGETS = {
    0: 800_000_000,    # Perairan  -> 800 juta
    1: 1_000_000_000,  # Hutan     -> 1 miliar
}

def subsample_to_targets(files, patch_dist, targets, seed=42):
    rng = random.Random(seed)

    # Kelompokkan patch berdasarkan kelas dominan (kelas dengan piksel terbanyak)
    groups = {c: [] for c in range(N_CLASSES)}
    for f in files:
        counts = patch_dist.get(str(f), [0] * N_CLASSES)
        groups[int(np.argmax(counts))].append(f)

    print("Rincian per kelas:")
    selected = []
    for cls in range(N_CLASSES):
        cls_files = list(groups[cls])
        if cls not in targets:
            selected.extend(cls_files)
            print(f"  {CLASS_NAMES[cls]:<18}: keep semua {len(cls_files):,} patches")
            continue

        target_px = targets[cls]
        rng.shuffle(cls_files)
        total, kept = 0, []
        for f in cls_files:
            counts = patch_dist.get(str(f), [0] * N_CLASSES)
            total += counts[cls]
            kept.append(f)
            if total >= target_px:
                break
        selected.extend(kept)
        status = ">= target" if total >= target_px else "< target (semua dipakai)"
        print(f"  {CLASS_NAMES[cls]:<18}: {len(cls_files):,} -> {len(kept):,} patches "
              f"| {total:,} px ({status})")

    return selected

print(f"Target  Perairan : {PIXEL_TARGETS[0]:,} px")
print(f"Target  Hutan    : {PIXEL_TARGETS[1]:,} px")
print()
selected_train = subsample_to_targets(train_files, patch_dist, PIXEL_TARGETS)
print(f"\nRingkasan: {len(train_files):,} -> {len(selected_train):,} patches "
      f"(-{100*(1-len(selected_train)/len(train_files)):.0f}%)")

Target  Perairan : 800,000,000 px
Target  Hutan    : 1,000,000,000 px

Rincian per kelas:
  Perairan          : 103,680 -> 12,319 patches | 800,022,913 px (>= target)
  Hutan             : 60,047 -> 16,663 patches | 1,000,044,793 px (>= target)
  Lahan Terbuka     : keep semua 774 patches
  Sawit             : keep semua 1,604 patches
  Pertanian Lain    : keep semua 3,675 patches
  Tambang           : keep semua 1,369 patches
  Permukiman        : keep semua 1,929 patches

Ringkasan: 173,078 -> 38,333 patches (-78%)


In [6]:
# === EDA SESUDAH normalisasi + estimasi training time ===

print_dist_table(selected_train, patch_dist, "SESUDAH normalisasi — Train (patch terpilih)")

# Perbandingan ringkas
batch_size        = 8
it_per_sec_jetson = 3.5   # estimasi konservatif Jetson AGX Orin (AMP FP16)
n_iter_before = len(train_files)    / batch_size
n_iter_after  = len(selected_train) / batch_size

print(f"\n{'='*52}")
print(f"  Perbandingan sebelum vs sesudah")
print(f"{'='*52}")
print(f"  {'':25} {'Sebelum':>10}  {'Sesudah':>10}")
print(f"  {'-'*48}")
print(f"  {'Total patches':25} {len(train_files):>10,}  {len(selected_train):>10,}")
print(f"  {'Iter per epoch':25} {n_iter_before:>10,.0f}  {n_iter_after:>10,.0f}")
print(f"  {'Estimasi menit/epoch':25} {n_iter_before/it_per_sec_jetson/60:>9.0f}m  "
      f"{n_iter_after/it_per_sec_jetson/60:>9.0f}m")
print(f"  {'Estimasi 40 epoch':25} {n_iter_before/it_per_sec_jetson/3600*40:>9.1f}j  "
      f"{n_iter_after/it_per_sec_jetson/3600*40:>9.1f}j")
print(f"{'='*52}")
print(f"  (asumsi {it_per_sec_jetson} it/s @ Jetson AGX Orin, batch {batch_size})")


  SESUDAH normalisasi — Train (patch terpilih)
  Total patches : 38,333
  Total piksel  : 2,512,191,488
  Kelas                          Piksel       %   Patch (dominan)
  --------------------------------------------------------------
  Perairan                  840,790,697   33.5%         12,319
  Hutan                   1,073,786,271   42.7%         16,663
  Lahan Terbuka              83,766,718    3.3%            774
  Sawit                      92,515,861    3.7%          1,604
  Pertanian Lain            239,372,582    9.5%          3,675
  Tambang                    77,111,358    3.1%          1,369
  Permukiman                104,848,001    4.2%          1,929
  --------------------------------------------------------------
  TOTAL                   2,512,191,488  100.0%         38,333

  Perbandingan sebelum vs sesudah
                               Sebelum     Sesudah
  ------------------------------------------------
  Total patches                173,078      38,333
  Iter 

In [7]:
# === Simpan daftar patch terpilih ke cache ===
# Disimpan sebagai arcname (portable lintas env): "papua/tile_xxx/p*.npz",
# "transfer/<kelas>/tile_xxx/p*.npz", "aug/p*.npz"

def to_arcname(f):
    """Konversi path absolut ke arcname portable (prefix papua/transfer/aug)."""
    if ENV == "jetson":
        # path: DATA_ROOT/train/papua/... atau DATA_ROOT/train/transfer/...
        try:
            rel = Path(f).relative_to(BAHAN_DIR / "train")
            return rel.as_posix()
        except ValueError:
            return Path(f).name
    else:
        f = Path(f)
        for root, prefix in [
            (PATCH_DIR,         "papua"),
            (PATCHES_TRANSFER,  "transfer"),
            (AUGMENTED_PATCHES, "aug"),
        ]:
            try:
                return f"{prefix}/{f.relative_to(root).as_posix()}"
            except ValueError:
                continue
        return f.name  # fallback

selected_arcnames = [to_arcname(f) for f in selected_train]

OUT_JSON = CACHE_DIR / "selected_train_patches.json"
OUT_JSON.write_text(json.dumps(selected_arcnames, indent=1), encoding="utf-8")
print(f"Disimpan : {OUT_JSON}")
print(f"Total    : {len(selected_arcnames):,} arcnames")
print()
print("Cara pakai di training notebook (cell 4, setelah final_train_files dibentuk):")
print("  _sel = set(json.load(open(BAHAN_DIR / 'optimize_cache' / 'selected_train_patches.json')))")
print("  final_train_files = [f for f in final_train_files")
print("                       if to_arcname(f) in _sel]")

Disimpan : /content/drive/MyDrive/Satria Data 3.0/Bahan_Training_Model/optimize_cache/selected_train_patches.json
Total    : 38,333 arcnames

Cara pakai di training notebook (cell 4, setelah final_train_files dibentuk):
  _sel = set(json.load(open(BAHAN_DIR / 'optimize_cache' / 'selected_train_patches.json')))
  final_train_files = [f for f in final_train_files
                       if to_arcname(f) in _sel]


In [8]:
# === Hitung ulang class weights dari patch terpilih ===
from forestwatch.training.metrics import median_frequency_weights

dist_fix = [0] * N_CLASSES
for f in selected_train:
    counts = patch_dist.get(str(f), [0] * N_CLASSES)
    for c in range(N_CLASSES):
        dist_fix[c] += counts[c]

class_weights_fix = median_frequency_weights(dict(enumerate(dist_fix)), n_classes=N_CLASSES)

print("Class weights BARU (dari patch terpilih):")
print(f"  {'Kelas':<18} {'Piksel':>16}   {'Weight Baru':>11}")
print(f"  {'-'*50}")
for c in range(N_CLASSES):
    print(f"  {CLASS_NAMES[c]:<18} {dist_fix[c]:>16,}   {class_weights_fix[c]:>11.4f}")


Class weights BARU (dari patch terpilih):
  Kelas                        Piksel   Weight Baru
  --------------------------------------------------
  Perairan                840,790,697        0.3000
  Hutan                 1,073,786,271        0.3000
  Lahan Terbuka            83,766,718        1.2520
  Sawit                    92,515,861        1.1330
  Pertanian Lain          239,372,582        0.4380
  Tambang                  77,111,358        1.3600
  Permukiman              104,848,001        1.0000


In [9]:
# === Bundle patch terpilih ke Bahan_Training_Fix ===
# Format SAMA dengan Bahan_Training_Model: train_part_*.tar + val.tar + test.tar
# + class_weights.json (direcalculate) + patch_sampler_weights_shared.json

from forestwatch.data.dataset import create_dataset_archives
from forestwatch.utils.io import save_json

if ENV == 'jetson':
    print("[jetson] bundling tidak didukung dari Jetson — jalankan di Colab/lab.")
else:
    BAHAN_FIX = DRIVE_ROOT / 'Bahan_Training_Fix'
    BAHAN_FIX.mkdir(parents=True, exist_ok=True)

    # Bangun arcname items dari train_files (pakai to_arcname yang sudah ada di cell 8)
    selected_set = set(selected_arcnames)

    def _all_items(files):
        """Konversi list path ke [(arcname, path)] pakai prefix papua/transfer/aug."""
        items = []
        for f in files:
            arc = to_arcname(f)
            items.append((arc, f))
        return items

    all_train_items = _all_items(train_files)
    train_items_fix = [(arc, src) for arc, src in all_train_items if arc in selected_set]

    # val/test tetap utuh (holdout tidak difilter)
    val_items  = [(f"papua/{Path(f).relative_to(PATCH_DIR).as_posix()}", f) for f in val_files]
    test_items = [(f"papua/{Path(f).relative_to(PATCH_DIR).as_posix()}", f) for f in test_files]

    splits_fix = {
        'train': train_items_fix,
        'val':   val_items,
        'test':  test_items,
    }

    print(f"train  : {len(train_files):,} -> {len(train_items_fix):,} patch "
          f"(-{100*(1 - len(train_items_fix)/len(train_files)):.0f}%)")
    print(f"val    : {len(val_items):,} patch (tidak difilter)")
    print(f"test   : {len(test_items):,} patch (tidak difilter)")
    create_dataset_archives(splits_fix, BAHAN_FIX, n_train_parts=7, max_workers=64)

    save_json({'class_weights': [float(w) for w in class_weights_fix]},
              BAHAN_FIX / 'class_weights.json')
    print(f"class_weights.json disimpan ke {BAHAN_FIX}")


train  : 173,078 -> 38,333 patch (-78%)
val    : 19,345 patch (tidak difilter)
test   : 19,346 patch (tidak difilter)


Bundling train_part01:   0%|          | 0/5477 [00:00<?, ?file/s]

Bundling train_part02:   0%|          | 0/5476 [00:00<?, ?file/s]

Bundling train_part03:   0%|          | 0/5476 [00:00<?, ?file/s]

Bundling train_part04:   0%|          | 0/5476 [00:00<?, ?file/s]

...

Bundling train_part06:   0%|          | 0/5476 [00:00<?, ?file/s]

Bundling train_part07:   0%|          | 0/5476 [00:00<?, ?file/s]

Bundling val_part01:   0%|          | 0/19345 [00:00<?, ?file/s]

Bundling test_part01:   0%|          | 0/19346 [00:00<?, ?file/s]

class_weights.json disimpan ke /content/drive/MyDrive/Satria Data 3.0/Bahan_Training_Fix


In [10]:
# === Hitung sampler weights untuk Bahan_Training_Fix + summary ===
import shutil
from forestwatch.data.dataset import compute_patch_sampler_weights

if ENV == 'jetson':
    print("[jetson] skip — jalankan di Colab/lab")
else:
    # Salin selected_train_patches.json ke optimize_cache dalam Bahan_Training_Fix
    _oc = BAHAN_FIX / 'optimize_cache'
    _oc.mkdir(exist_ok=True)
    shutil.copy(OUT_JSON, _oc / OUT_JSON.name)
    print(f"selected_train_patches.json disalin ke {_oc}")

    SAMPLER_CACHE_FIX = BAHAN_FIX / 'patch_sampler_weights_shared.json'
    if SAMPLER_CACHE_FIX.exists():
        print(f"[skip] {SAMPLER_CACHE_FIX.name} sudah ada.")
    else:
        sampler_files_fix = [src for _, src in train_items_fix]
        sampler_keys_fix  = ["/".join((Path("train") / arc).parts[-3:])
                             for arc, _ in train_items_fix]
        compute_patch_sampler_weights(
            sampler_files_fix, class_weights_fix,
            cache_path=SAMPLER_CACHE_FIX,
            keys=sampler_keys_fix,
        )
        print(f"{SAMPLER_CACHE_FIX.name} dihitung & disimpan.")

    print(f"\nBahan_Training_Fix siap di: {BAHAN_FIX}")
    print("Isi:")
    for item in sorted(BAHAN_FIX.iterdir()):
        if item.is_file():
            sz = item.stat().st_size
            sz_str = f"{sz/1e9:.2f} GB" if sz > 1e8 else f"{sz/1e6:.1f} MB"
            print(f"  {item.name:<45} {sz_str}")
        elif item.is_dir():
            print(f"  {item.name}/")
    print("\nNotebook training otomatis memakai Bahan_Training_Fix ini (cell 5).")


selected_train_patches.json disalin ke /content/drive/MyDrive/Satria Data 3.0/Bahan_Training_Fix/optimize_cache


Sampler weights:   0%|          | 0/38333 [00:00<?, ?it/s]

patch_sampler_weights_shared.json dihitung & disimpan.

Bahan_Training_Fix siap di: /content/drive/MyDrive/Satria Data 3.0/Bahan_Training_Fix
Isi:
  class_weights.json                            0.0 MB
  optimize_cache/
  patch_sampler_weights_shared.json             1.8 MB
  test/
  train/
  val/

Notebook training otomatis memakai Bahan_Training_Fix ini (cell 5).
